# Fun Bitmask Aside

Astro 128/256 (UC Berkeley, 2026)

Bitmasks are a compact way to store multiple boolean flags in a single integer. They are ubiquitous in astronomical survey data (SDSS, APOGEE, Gaia, JWST, etc.) for encoding quality flags, targeting information, and processing status.

## Quick Review: Binary Representation

Any non-negative integer can be represented in binary (base 2). Each binary digit ("bit") is either 0 or 1:

| Decimal | Binary | Meaning |
|---------|--------|---------|
| 0 | `0000` | No flags set |
| 1 | `0001` | Bit 0 set |
| 2 | `0010` | Bit 1 set |
| 4 | `0100` | Bit 2 set |
| 5 | `0101` | Bits 0 and 2 set |
| 9 | `1001` | Bits 0 and 3 set |

The key insight: each power of 2 corresponds to exactly one bit position, so a single integer can encode **many independent boolean flags** simultaneously.

In [1]:
# Python makes it easy to see binary representations
for val in [0, 1, 2, 4, 5, 9, 255, 4096]:
    print(f"{val:5d} = {val:016b} (binary)")

    0 = 0000000000000000 (binary)
    1 = 0000000000000001 (binary)
    2 = 0000000000000010 (binary)
    4 = 0000000000000100 (binary)
    5 = 0000000000000101 (binary)
    9 = 0000000000001001 (binary)
  255 = 0000000011111111 (binary)
 4096 = 0001000000000000 (binary)


## Bitwise Operations

Python provides bitwise operators that work on individual bits:

| Operator | Name | Description |
|----------|------|-------------|
| `&` | AND | 1 only if both bits are 1 |
| `\|` | OR | 1 if either bit is 1 |
| `^` | XOR | 1 if bits differ |
| `~` | NOT | Flips all bits |
| `<<` | Left shift | Shifts bits left (multiply by 2) |
| `>>` | Right shift | Shifts bits right (divide by 2) |

In [2]:
# Check if a specific bit is set
flag = 13  # binary: 1101 -> bits 0, 2, 3 are set

print(f"flag = {flag} = {flag:04b} (binary)")
print()

for bit in range(4):
    is_set = bool(flag & (1 << bit))  # shift 1 to the bit position, then AND
    print(f"  Bit {bit} (value {2**bit:2d}): {'SET' if is_set else '---'}")

flag = 13 = 1101 (binary)

  Bit 0 (value  1): SET
  Bit 1 (value  2): ---
  Bit 2 (value  4): SET
  Bit 3 (value  8): SET


## Astronomy Example: APOGEE Pixel Bitmasks

In APOGEE spectra, each pixel has a bitmask where different bits indicate different quality issues:

| Bit | Flag Name | Meaning |
|-----|-----------|--------|
| 0 | BADPIX | Pixel marked as bad |
| 1 | CRPIX | Cosmic ray hit |
| 2 | SATPIX | Saturated pixel |
| 3 | UNFIXABLE | Pixel marked as unfixable |
| 4 | BADDARK | Bad dark current |
| 5 | BADFLAT | Bad flat field |
| 6 | BADERR | Large uncertainty |
| 7 | NOSKY | No sky available |
| 12 | PERSIST_HIGH | High persistence |

See the [SDSS bitmask documentation](https://www.sdss4.org/dr17/algorithms/bitmasks/) for full details.

A pixel with bitmask value 9 (= `1001` in binary) means bits 0 and 3 are set: the pixel is both **BADPIX** and **UNFIXABLE**.

In [3]:
import numpy as np

# Simulate some APOGEE-like bitmask values
bitmask_values = np.array([0, 1, 4, 9, 4096, 4097, 0, 0, 2, 0])

# The bits we care about for Lab 2
target_bits = [0, 1, 2, 3, 4, 5, 6, 7, 12]
target_mask = np.sum(2**np.array(target_bits))

print(f"Target mask = {target_mask} = {target_mask:016b} (binary)")
print()

# Use bitwise AND to check if ANY of our target bits are set
bad_pixels = (bitmask_values & target_mask) != 0

for i, (val, bad) in enumerate(zip(bitmask_values, bad_pixels)):
    status = "BAD" if bad else "OK "
    print(f"  Pixel {i}: bitmask={val:5d} ({val:016b}) -> {status}")

Target mask = 4351 = 0001000011111111 (binary)

  Pixel 0: bitmask=    0 (0000000000000000) -> OK 
  Pixel 1: bitmask=    1 (0000000000000001) -> BAD
  Pixel 2: bitmask=    4 (0000000000000100) -> BAD
  Pixel 3: bitmask=    9 (0000000000001001) -> BAD
  Pixel 4: bitmask= 4096 (0001000000000000) -> BAD
  Pixel 5: bitmask= 4097 (0001000000000001) -> BAD
  Pixel 6: bitmask=    0 (0000000000000000) -> OK 
  Pixel 7: bitmask=    0 (0000000000000000) -> OK 
  Pixel 8: bitmask=    2 (0000000000000010) -> BAD
  Pixel 9: bitmask=    0 (0000000000000000) -> OK 


## Why Bitmasks Instead of Multiple Boolean Columns?

1. **Storage efficiency**: One 32-bit integer stores 32 independent flags. For a survey with millions of spectra and thousands of pixels each, this saves significant disk space.

2. **I/O speed**: Reading one integer column is faster than reading 32 boolean columns from a FITS file.

3. **Flexible querying**: Bitwise operations let you quickly check any combination of flags in a single operation.

4. **Historical convention**: FITS files and astronomical catalogs have used bitmasks since the 1980s. You'll encounter them everywhere!

In [4]:
# Practical tip: numpy makes it easy to work with bitmasks on arrays
np.random.seed(42)
n_pixels = 1000

# Simulate bitmask array (most pixels are fine, some have issues)
fake_bitmask = np.zeros(n_pixels, dtype=int)
fake_bitmask[np.random.choice(n_pixels, 50, replace=False)] = 1     # BADPIX
fake_bitmask[np.random.choice(n_pixels, 20, replace=False)] |= 2    # CRPIX (can overlap!)
fake_bitmask[np.random.choice(n_pixels, 10, replace=False)] |= 4096 # PERSIST_HIGH

# Count pixels with any bad flag
any_bad = (fake_bitmask & target_mask) != 0
print(f"Total pixels: {n_pixels}")
print(f"Pixels with any quality flag: {any_bad.sum()}")
print(f"Pixels with BADPIX (bit 0): {((fake_bitmask & 1) != 0).sum()}")
print(f"Pixels with CRPIX (bit 1): {((fake_bitmask & 2) != 0).sum()}")
print(f"Pixels with PERSIST_HIGH (bit 12): {((fake_bitmask & 4096) != 0).sum()}")

Total pixels: 1000
Pixels with any quality flag: 77
Pixels with BADPIX (bit 0): 50
Pixels with CRPIX (bit 1): 20
Pixels with PERSIST_HIGH (bit 12): 10
